In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:

df = pd.read_csv("../../../data/processed/patient_splits.csv")

training_df = pd.read_csv(
    "../../../data/raw/training_data.csv"
)

splits_df = pd.read_csv(
    "../../../data/processed/patient_splits.csv"
)

print(training_df.shape)
print(splits_df.shape)

training_df.head()

(942, 23)
(942, 2)


,Patient ID,Recording locations:,Age,Sex,Height,Weight,Pregnancy status,Murmur,Murmur locations,Most audible location,...,Systolic murmur pitch,Systolic murmur quality,Diastolic murmur timing,Diastolic murmur shape,Diastolic murmur grading,Diastolic murmur pitch,Diastolic murmur quality,Outcome,Campaign,Additional ID
0,2530,AV+PV+TV+MV,Child,Female,98.0,15.9,False,Absent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Abnormal,CC2015,NaN
1,9979,AV+PV+TV+MV,Child,Female,103.0,13.1,False,Present,AV+MV+PV+TV,TV,...,High,Harsh,NaN,NaN,NaN,NaN,NaN,Abnormal,CC2015,NaN
2,9983,AV+PV+TV+MV,Child,Male,115.0,19.1,False,Unknown,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Abnormal,CC2015,NaN
3,13918,AV+PV+TV+MV,Child,Male,98.0,15.9,False,Present,TV,TV,...,Low,Blowing,NaN,NaN,NaN,NaN,NaN,Abnormal,CC2015,NaN
4,14241,AV+PV+TV+MV,Child,Male,87.0,11.2,False,Present,AV+MV+PV+TV,PV,...,Low,Harsh,NaN,NaN,NaN,NaN,NaN,Abnormal,CC2015,NaN


In [9]:
# Extract useful columns

meta_df = training_df[  
    [
        "Patient ID",
        "Outcome"
    ]
].copy()

meta_df["Outcome"] = (
    meta_df["Outcome"]
    .map(
        {
            "Normal": 0,
            "Abnormal": 1
        }
    )
)

meta_df.head()

,Patient ID,Outcome
0,2530,1
1,9979,1
2,9983,1
3,13918,1
4,14241,1


In [10]:
# Merge tables on patient ID

meta_df = meta_df.merge(
    splits_df,
    on="Patient ID",
    how="inner"
)

print("Shape of meta_df:", meta_df.shape)
print(meta_df.head())


Shape of meta_df: (942, 3)
   Patient ID  Outcome  split
0        2530        1  train
1        9979        1    val
2        9983        1  train
3       13918        1  train
4       14241        1    val


---

In [ ]:
# Scan audio files

audio_dir = Path(
    "../../../data/raw/training_data"
)

audio_files = list(
    audio_dir.glob("*.wav")
)

print(
    f"{len(audio_files)} wav files found"
)

3163 wav files found


In [ ]:
# Patient -> Valve mapping
# Prepare df with appropriate columms


patient_audio = {}

for file in audio_files:

    filename = file.stem

    patient_id = filename.split("_")[0]

    if patient_id not in patient_audio:

        patient_audio[patient_id] = {
            "AV": [],
            "MV": [],
            "PV": [],
            "TV": []
        }

    if "_AV" in filename:
        patient_audio[patient_id]["AV"].append(
            str(file)
        )

    elif "_MV" in filename:
        patient_audio[patient_id]["MV"].append(
            str(file)
        )

    elif "_PV" in filename:
        patient_audio[patient_id]["PV"].append(
            str(file)
        )

    elif "_TV" in filename:
        patient_audio[patient_id]["TV"].append(
            str(file)
        )

In [ ]:
# Scan folder and fill up .wav files

rows = []

for _, row in meta_df.iterrows():

    patient_id = str(
        row["Patient ID"]
    )

    audio_info = patient_audio.get(
        patient_id,
        {
            "AV": [],
            "MV": [],
            "PV": [],
            "TV": []
        }
    )

    rows.append(
        {
            "Patient ID": patient_id,
            "Outcome": row["Outcome"],
            "split": row["split"],
            "AV": audio_info["AV"],
            "MV": audio_info["MV"],
            "PV": audio_info["PV"],
            "TV": audio_info["TV"]
        }
    )

cnn_df = pd.DataFrame(
    rows
)

cnn_df.head()

,Patient ID,Outcome,split,AV,MV,PV,TV
0,2530,1,train,[..\..\..\data\raw\training_data\2530_AV.wav],[..\..\..\data\raw\training_data\2530_MV.wav],[..\..\..\data\raw\training_data\2530_PV.wav],[..\..\..\data\raw\training_data\2530_TV.wav]
1,9979,1,val,[..\..\..\data\raw\training_data\9979_AV.wav],[..\..\..\data\raw\training_data\9979_MV.wav],[..\..\..\data\raw\training_data\9979_PV.wav],[..\..\..\data\raw\training_data\9979_TV.wav]
2,9983,1,train,[..\..\..\data\raw\training_data\9983_AV.wav],[..\..\..\data\raw\training_data\9983_MV.wav],[..\..\..\data\raw\training_data\9983_PV.wav],[..\..\..\data\raw\training_data\9983_TV.wav]
3,13918,1,train,[..\..\..\data\raw\training_data\13918_AV.wav],[..\..\..\data\raw\training_data\13918_MV.wav],[..\..\..\data\raw\training_data\13918_PV.wav],[..\..\..\data\raw\training_data\13918_TV.wav]
4,14241,1,val,[..\..\..\data\raw\training_data\14241_AV.wav],[..\..\..\data\raw\training_data\14241_MV.wav],[..\..\..\data\raw\training_data\14241_PV.wav],[..\..\..\data\raw\training_data\14241_TV.wav]


In [ ]:
# Examine patient splits

print(
    "Patients:",
    len(cnn_df)
)

print(
    "Train:",
    (cnn_df["split"] == "train").sum()
)

print(
    "Val:",
    (cnn_df["split"] == "val").sum()
)

print(
    "Test:",
    (cnn_df["split"] == "test").sum()
)

Patients: 942
Train: 659
Val: 141
Test: 142


In [19]:
# Examine some random patient

example = cnn_df.iloc[200]

print(
    "Patient:",
    example["Patient ID"]
)

print(
    "Outcome:",
    example["Outcome"]
)

print(
    "AV:",
    example["AV"]
)

print(
    "MV:",
    example["MV"]
)

print(
    "PV:",
    example["PV"]
)

print(
    "TV:",
    example["TV"]
)

Patient: 50204
Outcome: 0
AV: ['..\\..\\..\\data\\raw\\training_data\\50204_AV.wav']
MV: ['..\\..\\..\\data\\raw\\training_data\\50204_MV_1.wav', '..\\..\\..\\data\\raw\\training_data\\50204_MV_2.wav']
PV: []
TV: []


In [20]:
# Save dataframe as a CSV

cnn_df.to_pickle(
    "../../../data/processed/cnn_patient_dataframe.pkl"
)

cnn_df.to_csv(
    "../../../data/processed/cnn_patient_dataframe.csv",
    index=False
)